# Task 3.1 - TensorFlow/Keras Model
**Skill Set Go EduTech - AI/ML Internship - Week 3**

**Goal:** build and train a TensorFlow/Keras neural network, then compare it with the PyTorch model from Task 2.1.

**Dataset:** same as Task 2.1 - Breast Cancer Wisconsin (30 numeric features, binary classification) - so the
comparison between frameworks is fair: same data, same split, same architecture shape, same hyperparameters.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(42)
np.random.seed(42)
os.makedirs("results", exist_ok=True)
print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

## 1. Dataset setup (identical to Task 2.1, for a fair comparison)

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit on training data only
X_val = scaler.transform(X_val)

print("Train:", X_train.shape, "| Val:", X_val.shape)
print("Class balance (val):", np.bincount(y_val))

## 2. Model definition
Same shape as the PyTorch model in Task 2.1: 30 -> 16 -> 8 -> 1, ReLU activations, sigmoid output for binary
classification. In Keras the sigmoid is part of the model (`activation="sigmoid"` on the last layer) rather than
folded into the loss function, which is the main structural difference from the PyTorch version.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 3. Training

In [ ]:
start = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    verbose=2,
)
train_time = time.time() - start
print(f"Training time: {train_time:.1f}s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history.history["loss"], label="Train")
ax[0].plot(history.history["val_loss"], label="Validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Binary cross-entropy"); ax[0].set_title("Loss per epoch")
ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history.history["accuracy"], label="Train")
ax[1].plot(history.history["val_accuracy"], label="Validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].set_title("Accuracy per epoch")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/keras_curves.png", dpi=150)
plt.show()

## 4. Evaluation and sample predictions

In [ ]:
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"Final validation loss: {val_loss:.4f} | Final validation accuracy: {val_acc:.4f}")

probs = model.predict(X_val[:10], verbose=0).squeeze()
sample = pd.DataFrame({
    "actual": y_val[:10],
    "predicted_prob_benign": probs.round(3),
    "predicted": (probs > 0.5).astype(int),
})
sample["correct"] = sample["actual"] == sample["predicted"]
sample

In [ ]:
keras_results = {
    "task": "3.1", "framework": "TensorFlow/Keras", "dataset": "Breast Cancer Wisconsin (same split as Task 2.1)",
    "model": "30-16-8-1 feed-forward, ReLU, sigmoid output",
    "loss": "binary_crossentropy", "optimizer": "Adam (lr=0.001)",
    "batch_size": 32, "epochs": 50,
    "final_train_loss": round(history.history["loss"][-1], 4),
    "final_val_loss": round(val_loss, 4),
    "val_accuracy": round(float(val_acc), 4),
    "train_time_sec": round(train_time, 1),
}
with open("results/keras_results.json", "w") as f:
    json.dump(keras_results, f, indent=2)
keras_results

## 5. Comparison notes: PyTorch (Task 2.1) vs. TensorFlow/Keras (this task)
This is the required deliverable for 3.1 ("TensorFlow notebook + comparison notes"). Fill in the PyTorch numbers
from your Task 2.1 `results/results.json`, then compare.

| | PyTorch (Task 2.1) | TensorFlow/Keras (this task) |
|---|---|---|
| Final validation accuracy | *(from 2.1 results.json)* | `val_accuracy` above |
| Final validation loss | *(from 2.1 results.json)* | `val_loss` above |
| Lines of code for training loop | ~15 (manual loop) | 1 (`model.fit(...)`) |
| Manual steps required | zero_grad, forward, loss, backward, step | none - handled by `.fit()` |

**My notes (write in your own words):**
- Which framework felt more explicit / more automatic, and where did that help or get in the way?
- Did accuracy/loss come out similar between the two? Any difference, and why might that be (random init, framework
  defaults)?
- Which would you reach for first for a quick prototype vs. for understanding exactly what's happening during training?

-